# BatchNorm

In [1]:
import torch 
import random
import torch.nn.functional as F
import matplotlib.pyplot as plt 
from IPython.display import Image
%matplotlib inline 
random.seed(42)

## 1. Model Definition

In [2]:
words = open('data/names.txt', 'r').read().splitlines()

In [3]:
words[:5]

['emma', 'olivia', 'ava', 'isabella', 'sophia']

In [4]:
chars = [chr(i) for i in range(ord('a'), ord('z')+1)]
stoi = {s:i+1 for i, s in enumerate(chars)} # map characters to numbers
stoi['.'] = 0 # special character indicating start and end
itos = {i:s for s, i in stoi.items()} # used for visualization

In [5]:
block_size = 3 # context length (number of characters)

def build_dataset(words):
    # build the dataset
    X, Y = [], []
    
    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch] # find index
            X.append(context)
            Y.append(ix)
    
            # add the char
            context = context[1:] + [ix]

    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

# split into training and testing sets
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

In [6]:
Xtr.shape, Xtr.dtype, Ytr.shape, Ytr.dtype

(torch.Size([182625, 3]), torch.int64, torch.Size([182625]), torch.int64)

## 2. Building the Neural Network

In [7]:
n_embd     = 10  # embedding dimension per character
n_hidden   = 200 # neurons in hidden layer

g = torch.Generator().manual_seed(2147483647)

C  = torch.randn((27, n_embd),                    generator=g, requires_grad=True)
w1 = torch.randn((block_size * n_embd, n_hidden), generator=g, requires_grad=True)
b1 = torch.randn(n_hidden,                        generator=g, requires_grad=True)
w2 = torch.randn((n_hidden, 27),                  generator=g, requires_grad=True)
b2 = torch.randn(27,                              generator=g, requires_grad=True)

parameters = [C, w1, b1, w2, b2]
print(f"Total parameters: {sum(p.nelement() for p in parameters):,}")

Total parameters: 11,897


## 3. Training the Neural Network

In [8]:
epoch         = 10000
batch_size    = 32
learning_rate = 0.1
lossi         = []

for i in range(epoch):
    # minibatch
    ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
    Xb, Yb = Xtr[ix], Ytr[ix]

    # forward pass
    emb    = C[Xb]
    h      = torch.tanh(emb.view(-1, block_size * n_embd) @ w1 + b1)
    logits = h @ w2 + b2
    loss   = F.cross_entropy(logits, Yb)

    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    # update
    for p in parameters:
        p.data += -learning_rate * p.grad

    # logging
    lossi.append(loss.log10().item())
    if i % 1000 == 0:
        print(f"step {i:>6}/{epoch} | loss: {loss.item():.4f}")

step      0/10000 | loss: 27.8817
step   1000/10000 | loss: 4.0136
step   2000/10000 | loss: 2.9950
step   3000/10000 | loss: 2.9579
step   4000/10000 | loss: 2.3792
step   5000/10000 | loss: 2.4390
step   6000/10000 | loss: 2.3733
step   7000/10000 | loss: 2.7908
step   8000/10000 | loss: 2.4839
step   9000/10000 | loss: 2.5200


In [9]:
@torch.no_grad()
def split_loss(split):
    x, y = {
        'train': (Xtr, Ytr),
        'val':   (Xdev, Ydev),
        'test':  (Xte, Yte),
    }[split]
    emb    = C[x]
    embcat = emb.view(emb.shape[0], -1)
    h      = torch.tanh(embcat @ w1 + b1)
    logits = h @ w2 + b2
    loss   = F.cross_entropy(logits, y)
    print(split, loss.item())

split_loss('train')
split_loss('val')

train 2.5730533599853516
val 2.5856711864471436


Finally, let's sample some names from the new model to see if there are any improvements from the bigram model: 

In [10]:
for _ in range(10):
    out = []
    context = [0] * block_size

    while True:
        # getting the probabilities
        emb = C[torch.tensor([context])] # embed the current 3 context characters
        h = torch.tanh(emb.view(1, -1) @ w1 + b1) # hidden layer, one row as this is only one example
        logits = h @ w2 + b2 # output layer
        probs = F.softmax(logits, dim=1) # convert to probabilities

        # get the next character
        ix = torch.multinomial(probs, num_samples=1, generator=g).item() # sample randomly from the output probabilities
        context = context[1:] + [ix] # update context
        out.append(ix)

        # stop when we hit end
        if ix == 0:
            break 

    print(''.join(itos[i] for i in out[:-1]))

aadya
jesia
emmi
chare
chary
migr
nye
mafira
ydk
yne
